# jina-embeddings-v3 — Local Demo

Runs fully on your desktop (CPU works; GPU/Apple Silicon is faster). The model is ~570M parameters, so the first run downloads roughly 2 GB from Hugging Face.

**What this notebook covers**
1. Load the model
2. Basic embeddings
3. Semantic search (query vs. passage adapters)
4. Task adapters (classification, text-matching, separation)
5. Matryoshka dimension truncation
6. Multilingual search
7. Long-document input (up to 8192 tokens)

**License note:** the open weights are CC BY-NC 4.0 (non-commercial). Commercial use needs Jina's API or a commercial license.

## 1. Install dependencies

In [1]:
# Run once. Restart the kernel afterwards if packages were upgraded.
%pip install -q -U sentence-transformers torch numpy einops

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.58.0 requires starlette>=0.40.0, but you have starlette 0.38.6 which is incompatible.
gradio 6.20.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.115.0 which is incompatible.
gradio 6.20.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.38.6 which is incompatible.
numba 0.65.1 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2. Load the model

In [2]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Pick the best available device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

# trust_remote_code is required: the model ships its own architecture code (XLM-RoBERTa + RoPE + LoRA adapters)
model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True, device=device)
model.max_seq_length = 8192  # model supports up to 8192 tokens
print("Loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

Using device: cpu


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/734k [00:00<?, ?B/s]

custom_st.py:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/492 [00:00<?, ?it/s]

AttributeError: 'XLMRobertaLoRA' object has no attribute 'all_tied_weights_keys'

## 3. Basic embeddings

The `task` argument selects a LoRA adapter. `text-matching` is a good default for general similarity.

In [3]:
sentences = [
    "The bank approved my home loan application.",
    "My mortgage request was accepted by the lender.",
    "The cat is sleeping on the sofa.",
]

emb = model.encode(sentences, task="text-matching", prompt_name="text-matching", normalize_embeddings=True)
print("Shape:", emb.shape)  # (3, 1024)

# Embeddings are normalized, so dot product == cosine similarity
sim = emb @ emb.T
np.set_printoptions(precision=3, suppress=True)
print(sim)

NameError: name 'model' is not defined

## 4. Semantic search (retrieval)

For retrieval, use **different adapters** for queries and documents:
- `retrieval.query` for the user query
- `retrieval.passage` for the documents being indexed

In [ ]:
documents = [
    "A mutual fund pools money from many investors to buy a diversified portfolio of stocks and bonds.",
    "An IPO is the first time a private company offers its shares to the public on a stock exchange.",
    "A demat account holds your shares and securities in electronic form instead of paper certificates.",
    "Term insurance pays a lump sum to the nominee if the policyholder dies during the policy term.",
    "A systematic investment plan (SIP) lets you invest a fixed amount in a mutual fund at regular intervals.",
    "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
]

doc_emb = model.encode(documents, task="retrieval.passage", prompt_name="retrieval.passage", normalize_embeddings=True)

def search(query, top_k=3):
    q = model.encode([query], task="retrieval.query", prompt_name="retrieval.query", normalize_embeddings=True)
    scores = (q @ doc_emb.T)[0]
    for rank, i in enumerate(np.argsort(-scores)[:top_k], 1):
        print(f"{rank}. [{scores[i]:.3f}] {documents[i]}")
    print()

search("How do I invest a small amount every month?")
search("How do I store shares digitally?")
search("What happens when a company goes public?")

## 5. Other task adapters

| Task | Use for |
|---|---|
| `retrieval.query` / `retrieval.passage` | Search / RAG |
| `text-matching` | Semantic similarity, deduplication |
| `classification` | Features for classifiers (sentiment, intent) |
| `separation` | Clustering, reranking-style separation |

In [ ]:
from sklearn.cluster import KMeans  # installed below if missing

texts = [
    "Stock markets rallied after the rate cut announcement.",
    "Sensex and Nifty closed higher on strong bank earnings.",
    "The football team won the championship in extra time.",
    "The striker scored a hat-trick in last night's match.",
    "The new phone has a faster chip and a better camera.",
    "The laptop launch features a brighter display and longer battery life.",
]

cl_emb = model.encode(texts, task="separation", prompt_name="separation", normalize_embeddings=True)
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(cl_emb)
for label, text in sorted(zip(labels, texts)):
    print(label, "|", text)

In [ ]:
# If the cell above fails with ModuleNotFoundError: scikit-learn, run this and re-run it
# %pip install -q scikit-learn

## 6. Matryoshka (MRL) — shrink the embedding size

The first N dimensions of the vector remain meaningful, so you can truncate and re-normalize to save storage and speed up search. Accuracy drops gradually as you go smaller.

In [ ]:
def truncate(vecs, dim):
    v = vecs[:, :dim]
    return v / np.linalg.norm(v, axis=1, keepdims=True)

query = "How do I invest a small amount every month?"
q_full = model.encode([query], task="retrieval.query", prompt_name="retrieval.query", normalize_embeddings=True)

for dim in [1024, 512, 256, 128, 64, 32]:
    scores = (truncate(q_full, dim) @ truncate(doc_emb, dim).T)[0]
    best = int(np.argmax(scores))
    print(f"dim={dim:>4} | top score={scores[best]:.3f} | best doc: {documents[best][:60]}...")

## 7. Multilingual search

Query in one language, find documents in another.

In [ ]:
multi_docs = [
    "Mutual funds pool money from many investors.",             # English
    "म्यूचुअल फंड कई निवेशकों का पैसा एकत्र करता है।",               # Hindi
    "மியூச்சுவல் ஃபண்டுகள் பல முதலீட்டாளர்களின் பணத்தை ஒருங்கிணைக்கின்றன.",  # Tamil
    "Les fonds communs de placement regroupent l'argent de nombreux investisseurs.",  # French
    "El gato duerme en el sofá.",                              # Spanish (unrelated)
]

md_emb = model.encode(multi_docs, task="retrieval.passage", prompt_name="retrieval.passage", normalize_embeddings=True)
q = model.encode(["What is a mutual fund?"], task="retrieval.query", prompt_name="retrieval.query", normalize_embeddings=True)
scores = (q @ md_emb.T)[0]
for i in np.argsort(-scores):
    print(f"[{scores[i]:.3f}] {multi_docs[i]}")

## 8. Long input (up to 8192 tokens)

In [ ]:
import time

long_text = (
    "A demat account lets investors hold shares and securities electronically. "
    "It is linked to a trading account and a bank account for settlement. "
) * 300  # roughly 6-7k tokens

tokens = len(model.tokenizer(long_text)["input_ids"])
print("Approx. tokens:", tokens)

t = time.time()
long_emb = model.encode([long_text], task="retrieval.passage", prompt_name="retrieval.passage", normalize_embeddings=True)
print(f"Embedded in {time.time() - t:.1f}s, shape: {long_emb.shape}")

q = model.encode(["how are shares held electronically?"], task="retrieval.query", prompt_name="retrieval.query", normalize_embeddings=True)
print("Similarity to query:", float(q @ long_emb.T))

## Notes and troubleshooting

- **Slow on CPU?** Long inputs are heavy. Lower `model.max_seq_length` (e.g. 512) for short texts, and batch with `batch_size=8`.
- **Out of memory?** Reduce `max_seq_length` and `batch_size`.
- **Remote-code / `transformers` errors?** The model uses custom code that can lag behind the newest `transformers` release. Try `%pip install -q "transformers<5"` and restart the kernel.
- **Hosted alternative:** Jina's API supports the same model plus `late_chunking`, which embeds a whole long document first and then splits it into chunk vectors that keep document context.
- **For a real RAG system:** chunk your documents (200-500 tokens), embed chunks with `retrieval.passage`, store vectors in FAISS, pgvector or Qdrant, and embed queries with `retrieval.query`.